In [1]:
import joblib
import faiss
from sentence_transformers import SentenceTransformer
from groq import Groq

## load pkl file 

In [2]:
chunks = joblib.load("chunks.pkl")
index = faiss.read_index("faiss_index.index")

In [3]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## using the groq api key

In [ ]:
client = Groq(api_key="") # here using api key 

## function for question embedding, retrival, prompt and response

In [37]:
def answer_question(question, k = 3):
    question_embedding = embedding_model.encode([question], convert_to_numpy=True)

    faiss.normalize_L2(question_embedding)

    distances, indices = index.search(question_embedding, k=k)

    retrieved = [chunks[i] for i in indices[0] if i != -1]
    if not retrieved:
        return "I couldn't find anything relevant in the document to answer that."

    context = "\n\n".join(retrieved)

    prompt = f"""
                You are an expert Python tutor.

                Answer the question using the provided context.

                If the answer is not present in the context, reply:
                "I don't know based on the uploaded document."

                Context:
                {context}

                Question:
                {question}

                Answer:
            """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content


## ask question from user

In [38]:
question = input("Enter the Question: ")
question

'file handling with examples'

## get response from model

In [39]:
print(answer_question(question))

File handling in Python involves various operations such as reading, writing, and manipulating files. Here are some examples:

**Reading Files:**

1. `read()`: Reads and returns the entire content of the file as a single string.
   - Syntax: `file.read()`
   - Example: `with open('file.txt', 'r') as f: content = f.read()`

2. `readline()`: Reads and returns a single line from the file.
   - Syntax: `file.readline()`
   - Example: `with open('file.txt', 'r') as f: line = f.readline()`

3. `readlines()`: Reads all lines of the file and returns them as a list of strings.
   - Syntax: `file.readlines()`
   - Example: `with open('file.txt', 'r') as f: lines = f.readlines()`

**Iterating a File:**

- A file object can be iterated line by line directly in a for loop, which is memory-efficient for large files.
  - Example: `with open('file.txt', 'r') as f: for line in f: print(line)`

**Writing Files:**

1. `write(string)`: Writes the given string to the file.
   - Syntax: `file.write(string)`